# 09.04_TF_9_species_Python

ESM2 embedding 和九物种 TF 输入整理。

- 当前文件：`analysis/09_ai_analysis/09.04_TF_9_species_Python.ipynb`
- 原始来源：`Codes/09.04_TF_9_species.ipynb`（旧编号仅用于溯源）。
- 运行内核：**python**。
- 导入依赖：`numpy`, `os`, `pandas`, `re`, `scanpy`。
- 当前编号与流程见 `docs/workflow.md`、`docs/code_index.md`。
- 仅更新整理版导读；原分析单元格、参数和顺序保持不变。原始 cell 索引在本文件中加 1。

**本文件说明：** 读取 ESM2 35M embedding，准备九物种 TranscriptFormer 输入；保留原候选 OG、下采样数量与随机种子。


## 1. 处理ESM OGs embedding

In [ ]:
import numpy as np
import pandas as pd
import os

# 已存在的 embedding 和 metadata 路径
npy_path = "/share/home/zhangze/zz/NeuralOrigin/Data/09.AI/ESM/matrices/esm2_t12_35M/candidate_ogs_esm2_t12_35M.species_specific_og_embeddings.npy"
metadata_path = "/share/home/zhangze/zz/NeuralOrigin/Data/09.AI/ESM/matrices/esm2_t12_35M/candidate_ogs_esm2_t12_35M.species_specific_og_embedding_metadata.tsv"

# 改为用户可写目录
output_dir = "/share/home/zhangze/zz/NeuralOrigin/Data/09.AI/TranscriptFormer/OG_embeddings_user"
os.makedirs(output_dir, exist_ok=True)

# 1. 加载 embedding 和 metadata
embeddings = np.load(npy_path, allow_pickle=True)
metadata = pd.read_csv(metadata_path, sep="\t")

# 2. 检查行数是否对应
if len(embeddings) != len(metadata):
    raise ValueError(f"Embedding rows {len(embeddings)} do not match metadata rows {len(metadata)}")

# 3. 构建 (OG, species) -> embedding 字典
og_embedding_dict = {}
for i, row in metadata.iterrows():
    key = (row["Orthogroup_ID"], row["species"])
    og_embedding_dict[key] = embeddings[i]

# 4. 保存字典和对应metadata
npy_output_path = os.path.join(output_dir, "candidate_ogs_species_specific_embeddings_dict.npy")
csv_output_path = os.path.join(output_dir, "candidate_ogs_species_specific_embeddings_metadata.csv")

np.save(npy_output_path, og_embedding_dict, allow_pickle=True)
metadata.to_csv(csv_output_path, index=False)

print("Saved embedding dict to:", npy_output_path)
print("Saved metadata CSV to:", csv_output_path)

## 2. 处理AnnData数据

In [ ]:
import os
import re
import pandas as pd
import numpy as np
import scanpy as sc


# =========================
# 0. Input and output paths
# =========================

input_files = {
    "Spla": "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/scOrthoGeneH5ad/Spla.OG.normalized.h5ad",
    "ClH23": "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/scOrthoGeneH5ad/ClH23.OG.normalized.h5ad",
    "HoH13": "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/scOrthoGeneH5ad/HoH13.OG.normalized.h5ad",
    "TrH2": "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/scOrthoGeneH5ad/TrH2.OG.normalized.h5ad",
    "TrH1": "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/scOrthoGeneH5ad/TrH1.OG.normalized.h5ad",
    "Auco": "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/scOrthoGeneH5ad/Auco.OG.normalized.h5ad",
    "Clhe": "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/scOrthoGeneH5ad/Clhe.OG.normalized.h5ad",
    "Neve": "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/scOrthoGeneH5ad/Neve.OG.normalized.h5ad",
    "Dare": "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/scOrthoGeneH5ad/Dare.OG.normalized.h5ad",
}

phylum_map = {
    "Spla": "Porifera",
    "ClH23": "Placozoa",
    "HoH13": "Placozoa",
    "TrH2": "Placozoa",
    "TrH1": "Placozoa",
    "Auco": "Cnidaria",
    "Clhe": "Cnidaria",
    "Neve": "Cnidaria",
    "Dare": "Bilateria",
}

display_order_map = {
    "Spla": 1,
    "ClH23": 2,
    "HoH13": 3,
    "TrH2": 4,
    "TrH1": 5,
    "Auco": 6,
    "Clhe": 7,
    "Neve": 8,
    "Dare": 9,
}

candidate_ogs = [
    "OG0000112",
    "OG0000036",
    "OG0000203",
    "OG0000133",
    "OG0000166",
    "OG0000260",
]

out_h5ad_dir = "/share/home/zhangze/zz/NeuralOrigin/Data/09.AI/TranscriptFormer/input_h5ad_downsampled"
os.makedirs(out_h5ad_dir, exist_ok=True)

summary_records = []
celltype_summary_records = []

random_seed = 2026
target_n_cells = 10000


# =========================
# 1. Helper functions
# =========================

def is_valid_og_id(x):
    """
    Keep only standard Orthogroup IDs such as OG0000112.
    Remove feature IDs such as gene-evm.model.ptg000171l.10.
    """
    return bool(re.match(r"^OG[0-9]+$", str(x)))


def stratified_downsample_obs(
    obs,
    group_col="CellTypes",
    target_n=10000,
    random_state=2026,
):
    """
    Stratified downsampling while preserving all cell types.

    Strategy:
    1. If n_cells <= target_n, keep all cells.
    2. If n_cells > target_n:
       - allocate samples approximately proportional to CellTypes abundance.
       - guarantee at least 1 cell for every CellTypes category.
       - if a cell type has fewer allocated cells than available, sample without replacement.
    """
    n_total = obs.shape[0]

    if n_total <= target_n:
        return obs.index.to_numpy()

    rng = np.random.default_rng(random_state)

    celltype_counts = obs[group_col].astype(str).value_counts()
    n_celltypes = len(celltype_counts)

    if target_n < n_celltypes:
        raise ValueError(
            f"target_n={target_n} is smaller than number of CellTypes={n_celltypes}. "
            "Increase target_n to preserve all cell types."
        )

    # proportional allocation
    raw_alloc = celltype_counts / celltype_counts.sum() * target_n
    alloc = np.floor(raw_alloc).astype(int)

    # guarantee at least 1 cell per cell type
    alloc[alloc < 1] = 1

    # do not allocate more than available
    alloc = np.minimum(alloc, celltype_counts)

    # adjust total allocation to exactly target_n if possible
    current_total = int(alloc.sum())

    if current_total < target_n:
        remaining = target_n - current_total

        # add remaining cells to cell types with available capacity
        capacity = celltype_counts - alloc
        frac_priority = (raw_alloc - np.floor(raw_alloc)).sort_values(ascending=False)

        # first use fractional priority
        for ct in frac_priority.index:
            if remaining <= 0:
                break
            if capacity.loc[ct] > 0:
                add_n = min(int(capacity.loc[ct]), remaining)
                alloc.loc[ct] += add_n
                remaining -= add_n

        # if still remaining, use largest capacity
        if remaining > 0:
            for ct in capacity.sort_values(ascending=False).index:
                if remaining <= 0:
                    break
                cap = int(celltype_counts.loc[ct] - alloc.loc[ct])
                if cap > 0:
                    add_n = min(cap, remaining)
                    alloc.loc[ct] += add_n
                    remaining -= add_n

    elif current_total > target_n:
        excess = current_total - target_n

        # remove excess from largest allocated groups, keeping at least 1
        for ct in alloc.sort_values(ascending=False).index:
            if excess <= 0:
                break
            removable = int(alloc.loc[ct] - 1)
            if removable > 0:
                remove_n = min(removable, excess)
                alloc.loc[ct] -= remove_n
                excess -= remove_n

    selected_indices = []

    for ct, n_select in alloc.items():
        ct_indices = obs.index[obs[group_col].astype(str) == ct].to_numpy()
        selected = rng.choice(ct_indices, size=int(n_select), replace=False)
        selected_indices.extend(selected.tolist())

    selected_indices = np.array(selected_indices)

    # shuffle final selected cells
    rng.shuffle(selected_indices)

    return selected_indices


# =========================
# 2. Process each species
# =========================

for species, path in input_files.items():
    print(f"\nProcessing {species}")
    print(f"Input: {path}")

    adata = sc.read_h5ad(path)

    # Ensure IDs are strings
    adata.var_names = adata.var_names.astype(str)
    adata.obs_names = adata.obs_names.astype(str)

    if "CellTypes" not in adata.obs.columns:
        raise ValueError(f"{species}: obs['CellTypes'] not found.")

    # -------------------------
    # Filter features: keep only valid OG IDs
    # -------------------------
    valid_og_mask = np.array([is_valid_og_id(x) for x in adata.var_names])
    n_vars_before = adata.n_vars
    n_valid_ogs = int(valid_og_mask.sum())
    n_removed_non_og = int(n_vars_before - n_valid_ogs)

    adata = adata[:, valid_og_mask].copy()

    # -------------------------
    # Downsample cells to 10000 while preserving all CellTypes
    # -------------------------
    n_cells_before = adata.n_obs
    celltypes_before = set(adata.obs["CellTypes"].astype(str).unique())

    selected_obs = stratified_downsample_obs(
        adata.obs,
        group_col="CellTypes",
        target_n=target_n_cells,
        random_state=random_seed + display_order_map[species],
    )

    adata = adata[selected_obs, :].copy()

    n_cells_after = adata.n_obs
    celltypes_after = set(adata.obs["CellTypes"].astype(str).unique())

    missing_celltypes_after_downsampling = sorted(celltypes_before - celltypes_after)

    if len(missing_celltypes_after_downsampling) > 0:
        raise RuntimeError(
            f"{species}: Some CellTypes were lost after downsampling: "
            f"{missing_celltypes_after_downsampling}"
        )

    # Make obs_names unique across species
    adata.obs_names = [f"{species}_{x}" for x in adata.obs_names.astype(str)]

    # -------------------------
    # Keep only necessary obs
    # -------------------------
    obs_new = pd.DataFrame(index=adata.obs_names)
    obs_new["CellTypes"] = adata.obs["CellTypes"].astype(str).values
    obs_new["species"] = species
    obs_new["phylum"] = phylum_map[species]
    obs_new["display_order"] = display_order_map[species]

    adata.obs = obs_new

    # -------------------------
    # Keep only necessary var
    # -------------------------
    var_new = pd.DataFrame(index=adata.var_names.astype(str))
    var_new["og_id"] = adata.var_names.astype(str)
    var_new["ensembl_id"] = adata.var_names.astype(str)
    var_new["is_candidate_OG"] = var_new["og_id"].isin(candidate_ogs)

    adata.var = var_new

    # -------------------------
    # Remove unnecessary stored objects
    # -------------------------
    # adata.uns.clear()
    # adata.obsm.clear()
    # adata.varm.clear()
    # adata.obsp.clear()
    # adata.varp.clear()
    # adata.layers.clear()

    # Directly use normalized adata.X
    adata.raw = None

    # -------------------------
    # Save compressed h5ad
    # -------------------------
    out_path = os.path.join(
        out_h5ad_dir,
        f"{species}.OG.TF_input.downsampled_10000.compressed.h5ad"
    )

    adata.write_h5ad(
        out_path,
        compression="gzip",
        compression_opts=4,
    )

    present = [og for og in candidate_ogs if og in adata.var_names]
    missing = [og for og in candidate_ogs if og not in adata.var_names]

    summary_records.append({
        "species": species,
        "phylum": phylum_map[species],
        "display_order": display_order_map[species],
        "input_path": path,
        "output_path": out_path,
        "n_cells_before": n_cells_before,
        "n_cells_after": n_cells_after,
        "n_vars_before": n_vars_before,
        "n_valid_OGs_after_filter": adata.n_vars,
        "n_removed_non_OG_features": n_removed_non_og,
        "n_CellTypes_before": len(celltypes_before),
        "n_CellTypes_after": len(celltypes_after),
        "lost_CellTypes": ",".join(missing_celltypes_after_downsampling),
        "n_candidate_OGs_present": len(present),
        "candidate_OGs_present": ",".join(present),
        "candidate_OGs_missing": ",".join(missing),
    })

    # CellTypes distribution after downsampling
    ct_counts = adata.obs["CellTypes"].value_counts()
    for ct, count in ct_counts.items():
        celltype_summary_records.append({
            "species": species,
            "CellTypes": ct,
            "n_cells_after_downsampling": int(count),
        })

    print(f"Saved compressed h5ad: {out_path}")
    print(f"Cells: {n_cells_before} -> {n_cells_after}")
    print(f"OG features: {n_vars_before} -> {adata.n_vars}")
    print(f"Removed non-OG features: {n_removed_non_og}")
    print(f"CellTypes retained: {len(celltypes_before)} -> {len(celltypes_after)}")
    print(f"Candidate OGs present: {present}")
    print(f"Candidate OGs missing: {missing}")


# =========================
# 3. Save summary tables
# =========================

summary_df = pd.DataFrame(summary_records)
summary_path = os.path.join(
    out_h5ad_dir,
    "TF_input_downsampled_10000_compressed_h5ad_summary.tsv"
)
summary_df.to_csv(summary_path, sep="\t", index=False)

celltype_summary_df = pd.DataFrame(celltype_summary_records)
celltype_summary_path = os.path.join(
    out_h5ad_dir,
    "TF_input_downsampled_10000_CellTypes_summary.tsv"
)
celltype_summary_df.to_csv(celltype_summary_path, sep="\t", index=False)

print("\nFinished.")
print(f"Summary saved to: {summary_path}")
print(f"CellTypes summary saved to: {celltype_summary_path}")

In [ ]:
import scanpy as sc

adata_Auco = sc.read_h5ad('/share/home/zhangze/zz/NeuralOrigin/Data/09.AI/TranscriptFormer/input_h5ad_downsampled/Auco.OG.TF_input.downsampled_10000.compressed.h5ad')
adata_Auco

In [ ]:
adata_Auco.obs

In [ ]:
adata_Auco.var